<a href="https://colab.research.google.com/github/juliajohanson/CP-02-PAI-28-04/blob/main/CP_02_PAI_28_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🟢 Servidor Ollama iniciado!

✅ Tudo pronto! Servidor rodando e modelo baixado.


In [62]:
# PRIMEIRA COMVERSA COM MODELO BASE

import ollama

# Llama 3.2 de 1B — leve e rápido para aprender
MODELO_BASE = "llama3.2:1b"

def perguntar(modelo, pergunta, system=None):
    mensagens = []
    if system:
        mensagens.append({"role": "system", "content": system})
    mensagens.append({"role": "user", "content": pergunta})

    resposta = ollama.chat(model=modelo, messages=mensagens)
    return resposta["message"]["content"]

# Testar sem personalização
pergunta = "Quanto devo receber ao passar novamente na casa 'Partida' (Go) após dar a volta no tabuleiro? Monopoly"
print("🤖 Modelo BASE (sem customização):")
print(perguntar(MODELO_BASE, pergunta))

🤖 Modelo BASE (sem customização):
Não tenho informações sobre uma casa chamada "Partida" em um jogo de Monopoly. O número exato de dinheiro que você recebe pode variar dependendo do quão bem jogou e da estratégia adotada, mas geralmente não é mais do que o montante inicial.


In [72]:
# Dataset: Assistente de regras de um jogo
dataset = [
    {
        "pergunta": "No Monopoly quanto devo dar a cada participante no começo do jogo se são 3 pessoas jogando?",
        "resposta": "Cada jogador recebe $1.500 no início da partida, independentemente do número de participantes, a distribuição deve ser feita de modo que cada jogador receba 2 notas de $500, 2 notas de $100, 2 notas de $50, 6 notas de $20, 5 notas de $10, 5 notas de $5, 5 notas de $1."
    },
    {
        "pergunta": "No jogo monopoly o que acontece se eu parar numa casa livre e não quiser comprar?",
        "resposta": "Se você cair em uma propriedade sem dono e não quiser comprá-la pelo valor de face, o banco deve colocar essa propriedade em leilão imediatamente. \n Todos os jogadores participam do leilão, inclusive você \n O leilão começa com qualquer valor (inclusive $1) e o jogador que der o maior lance leva a propriedade."
    },
    {
        "pergunta": "No jogo Monopoly, o que acontece se eu parar em uma casa que já é propiedade de outro jogador?",
        "resposta": "Se você parar em uma propriedade que já é de outro jogador, deve pagar o aluguel ao proprietário, conforme o valor indicado na carta de título da propriedade"
    },
    {
        "pergunta": "Como faço para ganhar o jogo? Monopoly",
        "resposta": "Para vencer no Monopoly, o objetivo é levar todos os oponentes à falência, priorizando a compra agressiva de propriedades e a formação de monopólios o mais cedo possível."
    },
    {
        "pergunta": "Ganha dinheiro se cair na Parada Livre no Monopoly?",
        "resposta": "Não. De acordo com as regras oficiais, a Parada Livre é apenas um espaço de descanso. Você não ganha dinheiro, taxas ou bônus ao cair nela. (Colocar dinheiro de multas lá é uma 'regra da casa' comum, mas não oficial)."
    },
    {
        "pergunta": "O que acontece oficialmente quando um jogador cai na casa 'Estacionamento Grátis'?",
        "resposta": "De acordo com as regras oficiais, o 'Estacionamento Grátis' é apenas um espaço de descanso onde nada acontece. O jogador não recebe dinheiro, taxas ou propriedades ao parar ali."
    },
    {
        "pergunta": "Quanto um jogador recebe do banco ao passar ou cair na casa 'Ponto de Partida'?",
        "resposta": "O jogador recebe $200 do banco toda vez que seu peão passar ou parar no espaço do 'Ponto de Partida'."
    },
    {
        "pergunta": "Quais são as formas permitidas de sair da Prisão no Monopoly?",
        "resposta": "Um jogador pode sair da prisão: 1) Tirando dados iguais (dobradinha) em sua jogada; 2) Usando a carta 'Saída Livre da Prisão'; ou 3) Pagando uma multa de $50 ao banco."
    },
    {
        "pergunta": "O que ocorre se um jogador tirar números iguais nos dados (dobradinha) três vezes seguidas no mesmo turno?",
        "resposta": "Se um jogador tirar três dobradinhas consecutivas, ele deve ir imediatamente para a Prisão, terminando o seu turno sem completar o movimento da terceira jogada."
    },
    {
        "pergunta": "Qual é o requisito para poder começar a comprar hotéis para suas propriedades?",
        "resposta": "O jogador deve possuir todas as propriedades de um mesmo grupo de cores e ter construído exatamente quatro casas em cada uma delas antes de poder trocá-las por um hotel."
    },
    {
        "pergunta": "O que acontece se o jogador tirar números iguais nos dados (dobradinha)?",
        "resposta": "O jogador move o seu peão normalmente, executa a ação da casa onde caiu e tem o direito de jogar os dados novamente para um novo movimento no mesmo turno."
    },
    {
        "pergunta": "Como funciona o aluguel de um terreno que faz parte de um grupo de cores completo, mas ainda não possui casas construídas?",
        "resposta": "Se um jogador possuir todas as propriedades de um grupo de cores, o aluguel cobrado em qualquer um desses terrenos sem construções é dobrado."
    },
    {
        "pergunta": "Qual é o valor do aluguel de uma Ferrovia se o dono possuir as 4 estações do tabuleiro?",
        "resposta": "O aluguel cobrado quando um jogador cai em qualquer uma das quatro ferrovias pertencentes ao mesmo dono é de $200."
    },
    {
        "pergunta": "É permitido pedir dinheiro emprestado a outro jogador para pagar uma dívida ou comprar propriedades?",
        "resposta": "Não. De acordo com as regras oficiais, os jogadores só podem pedir empréstimos ao Banco, hipotecando suas propriedades. Empréstimos entre jogadores são proibidos."
    },
    {
        "pergunta": "O que acontece com as casas e hotéis de uma propriedade se o jogador decidir hipotecá-la?",
        "resposta": "Antes de hipotecar uma propriedade, o jogador deve obrigatoriamente vender todas as casas e hotéis daquele grupo de cores de volta para o Banco pela metade do preço."
    },
    {
        "pergunta": "É possível vencer o Monopoly apenas acumulando uma certa quantia de dinheiro?",
        "resposta": "Não. Nas regras oficiais, não existe um valor fixo de dinheiro que garanta a vitória; o jogo só termina quando resta apenas um jogador solvente e todos os outros faliram."
    },
    {
        "pergunta": "Um jogador pode ter 4 casas e 1 hotel no mesmo terreno simultaneamente?",
        "resposta": "Não. Quando um jogador constrói um hotel, ele deve devolver as 4 casas daquela propriedade para o banco. O hotel substitui as casas."
    },
    {
        "pergunta": "O que define a falência de um jogador no Monopoly?",
        "resposta": "A falência ocorre quando um jogador deve mais do que pode pagar ao banco ou a outro jogador, mesmo após vender suas casas/hotéis e hipotecar todas as suas propriedades."
    },
    {
        "pergunta": "Se o tempo de jogo for limitado (regra de tempo), como se define o vencedor?",
        "resposta": "Nesse caso específico, cada jogador soma seu dinheiro, o valor de face de suas propriedades, o valor de propriedades hipotecadas e o preço de custo de suas casas/hotéis. Quem tiver o maior patrimônio total vence."
    },
    {
        "pergunta": "Ter o monopólio de uma cor (todas as propriedades do grupo) é obrigatório para vencer?",
        "resposta": "Não é uma regra de vitória, mas é uma estratégia essencial, pois somente com o grupo completo é permitido construir casas e hotéis para cobrar aluguéis mais altos."
    }
]

print(f"✅ Dataset criado com {len(dataset)} exemplos")

✅ Dataset criado com 20 exemplos


In [54]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    """Gera o conteúdo de um Modelfile para o Ollama."""
    linhas = []

    # 1. Modelo base
    linhas.append(f"FROM {modelo_base}\n")

    # 2. Parâmetros de geração
    linhas.append("PARAMETER temperature 0.3")
    linhas.append("PARAMETER num_predict 300\n")

    # 3. System prompt permanente
    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    # 4. Exemplos de conversa (few-shot embutido no modelo)
    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)
SYSTEM_PROMPT = """
Você é um especialista rigoroso nas regras oficiais do Monopoly (Hasbro).

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável e objetivo
- Use emojis relevantes para humanizar a conversa
- Nunca prometa algo que não possa cumprir
- Foque em responder a duvida do cliente rapidamente e objetivamente

REGRAS CRÍTICAS:

1. A única condição de vitória é ser o último jogador restante após a falência de todos os outros.
2. NÃO existe vitória por acúmulo de patrimônio, valor de $1.500 ou compra de todas as propriedades.
3. Se o usuário sugerir regras erradas, corrija-as gentilmente baseando-se no manual oficial.
"""

conteudo_modelfile = gerar_modelfile(MODELO_BASE, SYSTEM_PROMPT, dataset)

print("📄 Modelfile gerado:\n")
print(conteudo_modelfile)

📄 Modelfile gerado:

FROM llama3.2:1b

PARAMETER temperature 0.3
PARAMETER num_predict 300

SYSTEM """

Você é um especialista rigoroso nas regras oficiais do Monopoly (Hasbro). 

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável e objetivo
- Use emojis relevantes para humanizar a conversa
- Nunca prometa algo que não possa cumprir
- Foque em responder a duvida do cliente rapidamente e objetivamente

REGRAS CRÍTICAS:

1. A única condição de vitória é ser o último jogador restante após a falência de todos os outros. 
2. NÃO existe vitória por acúmulo de patrimônio, valor de $1.500 ou compra de todas as propriedades.
3. Se o usuário sugerir regras erradas, corrija-as gentilmente baseando-se no manual oficial.

"""

MESSAGE user "No Monopoly quanto devo dar a cada participante no começo do jogo se são 3 pessoas jogando?"
MESSAGE assistant "Cada jogador recebe $1.500 no início da partida, independentemente do número de participantes, a distribuição deve ser feita de mo

In [55]:
# Salvar o Modelfile em disco
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo_modelfile)

# Criar o modelo customizado via CLI do Ollama
# O nome do nosso modelo personalizado será "suporte-monopoly"
!ollama create suporte-monopoly -f Modelfile

print("✅ Modelo customizado 'suporte-monopoly' criado!")


✅ Modelo customizado 'suporte-monopoly' criado!


In [71]:
MODELO_CUSTOM = "suporte-monopoly"

# Perguntas inéditas — NÃO estavam no dataset de treino
testes = [
    "Quantas propriedades de uma mesma cor um jogador precisa ter para começar a construir casas?",
    "Os jogadores podem realizar trocas ou negociações de propriedades entre si?",
    "Posso comprar propriedades já na minha primeira jogada?",
    "É permitido comprar casas durante o turno de outro jogador?",
    "Tenho permissão para rolar os dados uma segunda vez se não estiver satisfeito com o resultado da minha rolagem?"
]

print("=" * 60)
print("🧪 TESTES — MODELO CUSTOMIZADO")
print("=" * 60)

for pergunta in testes:
    resposta = perguntar(MODELO_CUSTOM, pergunta)
    print(f"\n👤 {pergunta}")
    print(f"🤖 {resposta}")
    print("-" * 60)

🧪 TESTES — MODELO CUSTOMIZADO

👤 Quantas propriedades de uma mesma cor um jogador precisa ter para começar a construir casas?
🤖 Um jogador precisa possuir todas as 6 propriedades da mesma cor antes de poder construir casas.
------------------------------------------------------------

👤 Os jogadores podem realizar trocas ou negociações de propriedades entre si?
🤖 Sim, os jogadores podem realizar trocas ou negociações de propriedades entre si. O objetivo é encontrar uma propriedade que seja melhor para o jogador em quem a propriedade é transferida.
------------------------------------------------------------

👤 Posso comprar propriedades já na minha primeira jogada?
🤖 Não. De acordo com as regras oficiais, um jogador pode comprar propriedades apenas após ter passado pelo 'Ponto de Partida' e ter recebido $200 do banco.
------------------------------------------------------------

👤 É permitido comprar casas durante o turno de outro jogador?
🤖 Não. O jogador que está fazendo a jogada pod

In [60]:
print("=" * 60)
print("🆚 BASE vs. CUSTOMIZADO")
print("=" * 60)

pergunta_comparacao = "Como faço para ganhar o jogo? Monopoly"
print(f"\n📌 Pergunta: {pergunta_comparacao}\n")

print("── Modelo BASE ──────────────────────")
print(perguntar(MODELO_BASE, pergunta_comparacao))

print("\n── Modelo CUSTOMIZADO ───────────────")
print(perguntar(MODELO_CUSTOM, pergunta_comparacao))

🆚 BASE vs. CUSTOMIZADO

📌 Pergunta: Como faço para ganhar o jogo? Monopoly

── Modelo BASE ──────────────────────
Ganhar no Monopoly pode ser um desafio emocionante! Aqui estão algumas dicas para ajudá-lo a aumentar suas chances de vencer:

1. **Conheça o jogo**: Antes de começar, é importante conhecer as regras do Monopoly e entender como cada jogador começa. Isso inclui saber quem é o proprietário do quarto lotado, quem é o arquiteto, e como são distribuídos os recursos (dinheiro, cartas, etc.).
2. **Planeje sua estratégia**: Antes de começar a jogar, planeje sua estratégia. Você pode querer:
 * Aquecer seu dinheiro rapidamente para investir em propriedades.
 * Comprar propriedades com alto potencial de aumentar o valor.
 * Usar cartas para ganhar dinheiro ou evitar perder dinheiro.
3. **Compre propriedades**: As propriedades são fundamentais no Monopoly, pois elas podem ser vendidas por uma grande quantia de dinheiro. Compre as propriedades que você pode vender mais facilmente e as 

In [64]:
def duvidas_monopoly():
    """Simula uma conversa completa com histórico de mensagens."""
    historico = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    perguntas_simuladas = [
        "Oi, tudo bem?",
        "Qual é o primeiro passo para começar uma partida de monopoly?",
        "Como compro propiedades conforme o avanço do jogo?",
        "O que determina o vencedor do jogo?",
    ]

    print("🗨️ Conversa com histórico:\n")

    for pergunta in perguntas_simuladas:
        historico.append({"role": "user", "content": pergunta})
        resposta = ollama.chat(model=MODELO_CUSTOM, messages=historico)
        conteudo = resposta["message"]["content"]
        historico.append({"role": "assistant", "content": conteudo})

        print(f"👤 {pergunta}")
        print(f"🤖 {conteudo}\n")
        print("-" * 50)

duvidas_monopoly()

🗨️ Conversa com histórico:

👤 Oi, tudo bem?
🤖 Tudo bem! Estou funcionando corretamente agora 😊

--------------------------------------------------
👤 Qual é o primeiro passo para começar uma partida de monopoly?
🤖 O primeiro passo para começar uma partida de Monopoly é distribuir os bens entre os jogadores. Cada jogador recebe $1.500, e os bens são divididos em propriedades, hotéis, casas e outros ativos.

--------------------------------------------------
👤 Como compro propiedades conforme o avanço do jogo?
🤖 Compre as propriedades ao comprar todos os bens de um grupo de cores. Você pode comprar todas as propriedades que pertencem a cada jogador ou apenas aquelas que não estão hipotecadas por outros jogadores.

--------------------------------------------------
👤 O que determina o vencedor do jogo?
🤖 O vencedor é o jogador que acumulará o maior patrimônio total ao finalizar o jogo, incluindo:

* Propriedades
* Casas e hotéis construídos
* Outros ativos (como terrenos hipotecados)

Quem

In [65]:
def chatbot_interativo(modelo, system_prompt):
    """Chatbot com input do usuário em tempo real."""
    historico = [
        {"role": "system", "content": system_prompt}
    ]

    print("=" * 60)
    print("💬 CHATBOT REGRAS MONOPOLY — MODO INTERATIVO")
    print("   Digite sua mensagem e pressione Enter.")
    print("   Para encerrar, digite: sair")
    print("=" * 60)

    while True:
        # Capturar input do usuário
        entrada = input("\n👤 Você: ").strip()

        # Verificar comando de saída
        if entrada.lower() == "sair":
            print("\n🤖 Assistente: Obrigado por entrar em contato! Até logo! 👋")
            print("=" * 60)
            print("👋 Sessão encerrada.")
            break

        # Ignorar mensagens vazias
        if not entrada:
            print("⚠️  Digite algo ou 'sair' para encerrar.")
            continue

        # Adicionar mensagem do usuário ao histórico
        historico.append({"role": "user", "content": entrada})

        # Chamar o modelo
        print("🤖 Assistente: ", end="", flush=True)
        resposta = ollama.chat(model=modelo, messages=historico)
        conteudo = resposta["message"]["content"]

        # Exibir e salvar resposta no histórico
        print(conteudo)
        historico.append({"role": "assistant", "content": conteudo})

# Iniciar o chatbot com o modelo customizado
chatbot_interativo(MODELO_CUSTOM, SYSTEM_PROMPT)

💬 CHATBOT REGRAS MONOPOLY — MODO INTERATIVO
   Digite sua mensagem e pressione Enter.
   Para encerrar, digite: sair

👤 Você: Como devo distribuir o dinheiro no começo do jogo?
🤖 Assistente: Cada jogador recebe $1.500 no início da partida.

👤 Você: Se eu parar em uma casa e não quiser comprar a propiedade, o que acontece?
🤖 Assistente: Quando você cair em uma casa sem dono, o banco deve colocar essa propriedade em leilão imediatamente. Todos os jogadores participam do leilão, incluindo você. O leilão começa com qualquer valor (inclusive $1) e o jogador que der o maior lance leva a propriedade.

👤 Você: sair

🤖 Assistente: Obrigado por entrar em contato! Até logo! 👋
👋 Sessão encerrada.
